In [2]:
import os

# Force the notebook's brain to move into the stat359 folder
os.chdir('/home/jupyter/stat359')

print(f"📍 New Location: {os.getcwd()}\n")

folders_to_check = ["data", "models", "instructor"]
all_good = True

for folder in folders_to_check:
    if os.path.exists(folder):
        print(f"✅ Found '{folder}' folder!")
    else:
        print(f"❌ Missing '{folder}' folder!")
        all_good = False

if all_good:
    print("\n🎉 Health Check Passed! You are good to go!")

📍 New Location: /home/jupyter/stat359

✅ Found 'data' folder!
✅ Found 'models' folder!
✅ Found 'instructor' folder!

🎉 Health Check Passed! You are good to go!


In [4]:
!poetry run python -m instructor.final_project.arithmetic_llm.generate_foundational_plaintext \
  --num-samples 100000 \
  --max-depth 4 \
  --num-range 1 20 \
  --invalid-rate 0.05 \
  --output-txt data/foundational_corpus.txt

!poetry run python -m instructor.final_project.arithmetic_llm.generate_instruction_corpus_mixed \
  --num-samples 20000 \
  --max-depth 4 \
  --num-range 1 20 \
  --invalid-rate 0 \
  --output-mixed data/instruction_corpus.txt

!poetry run python -m instructor.final_project.arithmetic_llm.generate_corpus \
  --instruction-only \
  --num-samples 1000 \
  --max-depth 4 \
  --output-instruction data/instruction_corpus_test.txt \
  --num-range 1 20 \
  --invalid-rate 0

Generating instruction corpus with 1000 samples...
Instruction corpus saved to: data/instruction_corpus_test.txt
Corpus generation complete!


In [5]:
!poetry run python -m instructor.final_project.arithmetic_llm.train_tokenizer \
  --corpus-path data/foundational_corpus.txt \
  --output-dir data/tokenizer \
  --vocab-size 1000

Training BPE tokenizer with vocabulary size 1000...
Corpus: data/foundational_corpus.txt
Building corpus: 200000it [00:08, 24653.48it/s]
BPE merges:  19%|████▉                     | 190/1000 [00:00<00:00, 2990.03it/s]
Saving tokenizer to: data/tokenizer

Tokenizer Statistics:
  Vocabulary size: 278
  BPE merge operations: 190
  Special tokens: <pad>, <unk>, <bos>, <eos>, <think>, </think>

Test encoding:
  Input: 5 + 10 - 3
  Encoded (with BOS/EOS): [178, 124, 8, 28, 11, 100, 179]
  Decoded: <bos> 5 + 10 - 3 <eos>
  Encoded (without BOS/EOS): [124, 8, 28, 11, 100]

Test encoding:
  Input: 12 - (4 + 2)
  Encoded (with BOS/EOS): [178, 49, 11, 4, 112, 8, 88, 6, 179]
  Decoded: <bos> 12 - ( 4 + 2 ) <eos>
  Encoded (without BOS/EOS): [49, 11, 4, 112, 8, 88, 6]

Test encoding:
  Input: ((7+3)-(3+5))
  Encoded (with BOS/EOS): [178, 4, 4, 148, 8, 100, 6, 11, 4, 100, 8, 124, 6, 6, 179]
  Decoded: <bos> ( ( 7 + 3 ) - ( 3 + 5 ) ) <eos>
  Encoded (without BOS/EOS): [4, 4, 148, 8, 100, 6, 11, 4, 10

In [ ]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_foundational_training \
  --corpus-path data/foundational_corpus.txt \
  --output-dir models/ \
  --tokenizer-path data/tokenizer \
  --num-epochs 10 \
  --max-seq-length 512 \
  --batch-size 16 \
  --device cuda


FOUNDATIONAL MODEL TRAINING

Corpus: data/foundational_corpus.txt
Tokenizer: data/tokenizer
Output directory: models/

Training Configuration:
  Learning rate: 0.0001
  Batch size: 16
  Epochs: 10
  Warmup steps: 1000
  Gradient clip: 1.0
  Save every: 1000 steps
  Device: cuda

Model Configuration:
  d_model: 256
  nhead: 8
  num_layers: 6
  dim_feedforward: 1024
  dropout: 0.1
  max_seq_length: 512

Training output directory: models/foundational_20260301_232626_388023
Configuration: {'learning_rate': 0.0001, 'batch_size': 16, 'num_epochs': 10, 'warmup_steps': 1000, 'gradient_clip': 1.0, 'save_every': 1000, 'eval_every': 500, 'device': 'cuda', 'lora_config': None}
Loading tokenizer...
Tokenizer vocabulary size: 278
Initializing model configuration...
Creating dataloaders...
Training batches: 11250
Validation batches: 1250
Initializing model...
Model parameters: 4,941,312

Starting training...

Epoch 1/10
Epoch 1:   9%| | 998/11250 [01:01<11:18, 15.11it/s, loss=1.27, avg_loss=2.61, lr

In [8]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/foundational_20260301_232626_388023/best_model.pt \
  --tokenizer-path data/tokenizer \
  --max-gen-length 512 \
  --num-samples 100 \
  --batch-size 1


MODEL EVALUATION

Model: models/foundational_20260301_232626_388023/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 100
  Max depth: 5
  Number range: 1 to 20
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 100 test expressions...
Generated 100 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/100 samples
Evaluated 2/100 samples
Evaluated 3/100 samples
Evaluated 4/100 samples
Evaluated 5/100 samples
Evaluated 6/100 samples
Evaluated 7/100 samples
Evaluated 8/100 samples
Evaluated 9/100 samples
Evaluated 10/100 samples
Evaluated 11/100 samples
Evaluated 12/100 samples
Evaluated 13/100 samples
Evaluated 14/100 samples
Evaluated 15/100 samples
Evaluated 16/100 samples
Evaluated 17/100 samples
Evaluated 18/100 samples
Evaluated 19/100 samples
Evaluated 20/100 samples
Evaluated 21/100 samples

In [10]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_instruction_training \
  --instruction-corpus-path data/instruction_corpus.txt \
  --output-dir models/ \
  --tokenizer-path data/tokenizer \
  --foundational-checkpoint models/foundational_20260301_232626_388023/final_model.pt \
  --num-epochs 10 \
  --device cuda


INSTRUCTION FINE-TUNING

Instruction corpus: data/instruction_corpus.txt
Tokenizer: data/tokenizer
Foundational checkpoint: models/foundational_20260301_232626_388023/final_model.pt
Output directory: models/

Training Configuration:
  Learning rate: 5e-05
  Batch size: 32
  Epochs: 10
  Warmup steps: 500
  Gradient clip: 1.0
  Save every: 500 steps
  Device: cuda

Fine-tuning output directory: models/instruction_20260302_025909_323029
Configuration: {'learning_rate': 5e-05, 'batch_size': 32, 'num_epochs': 10, 'warmup_steps': 500, 'gradient_clip': 1.0, 'save_every': 500, 'eval_every': 500, 'device': 'cuda', 'lora_config': None}
Loading tokenizer...
Tokenizer vocabulary size: 278
Initializing model architecture...
Creating dataloaders...
Training batches: 1125
Validation batches: 125
Loading foundational model from: models/foundational_20260301_232626_388023/final_model.pt
Loaded checkpoint from epoch 10, step 112500
Model parameters: 4,941,312

Starting instruction fine-tuning...

Epoc

In [11]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260302_025909_323029/best_model.pt \
  --tokenizer-path data/tokenizer \
  --max-gen-length 512 \
  --batch-size 1 \
  --num-samples 1000


MODEL EVALUATION

Model: models/instruction_20260302_025909_323029/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 1000
  Max depth: 5
  Number range: 1 to 20
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 1000 test expressions...
Generated 1000 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/1000 samples
Evaluated 2/1000 samples
Evaluated 3/1000 samples
Evaluated 4/1000 samples
Evaluated 5/1000 samples
Evaluated 6/1000 samples
Evaluated 7/1000 samples
Evaluated 8/1000 samples
Evaluated 9/1000 samples
Evaluated 10/1000 samples
Evaluated 11/1000 samples
Evaluated 12/1000 samples
Evaluated 13/1000 samples
Evaluated 14/1000 samples
Evaluated 15/1000 samples
Evaluated 16/1000 samples
Evaluated 17/1000 samples
Evaluated 18/1000 samples
Evaluated 19/1000 samples
Evaluated 20/1000 samples
Ev

In [4]:
!poetry run python -m instructor.final_project.arithmetic_llm.generate_instruction_corpus_mixed \
  --num-samples 50000 \
  --max-depth 4 \
  --num-range 1 20 \
  --invalid-rate 0 \
  --output-mixed data/instruction_corpus_50k.txt

In [5]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_instruction_training \
  --instruction-corpus-path data/instruction_corpus_50k.txt \
  --output-dir models/ \
  --tokenizer-path data/tokenizer \
  --foundational-checkpoint models/foundational_20260301_232626_388023/final_model.pt \
  --num-epochs 10 \
  --device cuda


INSTRUCTION FINE-TUNING

Instruction corpus: data/instruction_corpus_50k.txt
Tokenizer: data/tokenizer
Foundational checkpoint: models/foundational_20260301_232626_388023/final_model.pt
Output directory: models/

Training Configuration:
  Learning rate: 5e-05
  Batch size: 32
  Epochs: 10
  Warmup steps: 500
  Gradient clip: 1.0
  Save every: 500 steps
  Device: cuda

Fine-tuning output directory: models/instruction_20260302_162638_395697
Configuration: {'learning_rate': 5e-05, 'batch_size': 32, 'num_epochs': 10, 'warmup_steps': 500, 'gradient_clip': 1.0, 'save_every': 500, 'eval_every': 500, 'device': 'cuda', 'lora_config': None}
Loading tokenizer...
Tokenizer vocabulary size: 278
Initializing model architecture...
Creating dataloaders...
Training batches: 2813
Validation batches: 313
Loading foundational model from: models/foundational_20260301_232626_388023/final_model.pt
Loaded checkpoint from epoch 10, step 112500
Model parameters: 4,941,312

Starting instruction fine-tuning...



In [3]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260302_162638_395697/best_model.pt \
  --tokenizer-path data/tokenizer \
  --max-gen-length 512 \
  --batch-size 1 \
  --num-samples 1000


MODEL EVALUATION

Model: models/instruction_20260302_162638_395697/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 1000
  Max depth: 5
  Number range: 1 to 20
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 1000 test expressions...
Generated 1000 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/1000 samples
Evaluated 2/1000 samples
Evaluated 3/1000 samples
Evaluated 4/1000 samples
Evaluated 5/1000 samples
Evaluated 6/1000 samples
Evaluated 7/1000 samples
Evaluated 8/1000 samples
Evaluated 9/1000 samples
Evaluated 10/1000 samples
Evaluated 11/1000 samples
Evaluated 12/1000 samples
Evaluated 13/1000 samples
Evaluated 14/1000 samples
Evaluated 15/1000 samples
Evaluated 16/1000 samples
Evaluated 17/1000 samples
Evaluated 18/1000 samples
Evaluated 19/1000 samples
Evaluated 20/1000 samples
Ev

In [ ]:
# upgrading foundational training
!poetry run python -m instructor.final_project.arithmetic_llm.run_foundational_training \
  --corpus-path data/foundational_corpus.txt \
  --output-dir models/ \
  --tokenizer-path data/tokenizer \
  --num-epochs 10 \
  --batch-size 16 \
  --device cuda \
  --d-model 512 \
  --num-layers 8



FOUNDATIONAL MODEL TRAINING

Corpus: data/foundational_corpus.txt
Tokenizer: data/tokenizer
Output directory: models/

Training Configuration:
  Learning rate: 0.0001
  Batch size: 16
  Epochs: 10
  Warmup steps: 1000
  Gradient clip: 1.0
  Save every: 1000 steps
  Device: cuda

Model Configuration:
  d_model: 512
  nhead: 8
  num_layers: 8
  dim_feedforward: 1024
  dropout: 0.1
  max_seq_length: 512

Training output directory: models/foundational_20260302_212649_321160
Configuration: {'learning_rate': 0.0001, 'batch_size': 16, 'num_epochs': 10, 'warmup_steps': 1000, 'gradient_clip': 1.0, 'save_every': 1000, 'eval_every': 500, 'device': 'cuda', 'lora_config': None}
Loading tokenizer...
Tokenizer vocabulary size: 278
Initializing model configuration...
Creating dataloaders...
Training batches: 11250
Validation batches: 1250
Initializing model...
Model parameters: 17,227,776

Starting training...

Epoch 1/10
Epoch 1:   9%| | 999/11250 [02:00<18:03,  9.46it/s, loss=1.5, avg_loss=2.16, lr

In [ ]:
# neeeddd new time stamp!!!

!poetry run python -m instructor.final_project.arithmetic_llm.run_instruction_training \
  --instruction-corpus-path data/instruction_corpus_50k.txt \
  --output-dir models/ \
  --tokenizer-path data/tokenizer \
  --foundational-checkpoint models/foundational_20260302_212649_321160/final_model.pt \
  --num-epochs 10 \
  --device cuda


INSTRUCTION FINE-TUNING

Instruction corpus: data/instruction_corpus_50k.txt
Tokenizer: data/tokenizer
Foundational checkpoint: models/foundational_20260302_212649_321160/final_model.pt
Output directory: models/

Training Configuration:
  Learning rate: 5e-05
  Batch size: 32
  Epochs: 10
  Warmup steps: 500
  Gradient clip: 1.0
  Save every: 500 steps
  Device: cuda

Fine-tuning output directory: models/instruction_20260303_013325_759230
Configuration: {'learning_rate': 5e-05, 'batch_size': 32, 'num_epochs': 10, 'warmup_steps': 500, 'gradient_clip': 1.0, 'save_every': 500, 'eval_every': 500, 'device': 'cuda', 'lora_config': None}
Loading tokenizer...
Tokenizer vocabulary size: 278
Initializing model architecture...
Creating dataloaders...
Training batches: 2813
Validation batches: 313
Loading foundational model from: models/foundational_20260302_212649_321160/final_model.pt
Loaded checkpoint from epoch 10, step 112500
Model parameters: 17,227,776

Starting instruction fine-tuning...


In [7]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260303_013325_759230/best_model.pt \
  --tokenizer-path data/tokenizer \
  --max-gen-length 512 \
  --batch-size 1 \
  --num-samples 1000


MODEL EVALUATION

Model: models/instruction_20260303_013325_759230/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 1000
  Max depth: 5
  Number range: 1 to 20
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 1000 test expressions...
Generated 1000 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/1000 samples
Evaluated 2/1000 samples
Evaluated 3/1000 samples
Evaluated 4/1000 samples
Evaluated 5/1000 samples
Evaluated 6/1000 samples
Evaluated 7/1000 samples
Evaluated 8/1000 samples
Evaluated 9/1000 samples
Evaluated 10/1000 samples
Evaluated 11/1000 samples
Evaluated 12/1000 samples
Evaluated 13/1000 samples
Evaluated 14/1000 samples
Evaluated 15/1000 samples
Evaluated 16/1000 samples
Evaluated 17/1000 samples
Evaluated 18/1000 samples
Evaluated 19/1000 samples
Evaluated 20/1000 samples
Ev

In [9]:
import json
import random
import os

# Updated path for your GCP environment
ood_dir = 'data/ood_tests'
os.makedirs(ood_dir, exist_ok=True)

def format_problem(expression, real_answer):
    """Formats the data exactly how the model's parser expects it."""
    return {
        "expression": expression,
        "problem": f"Evaluate: {expression} <think>",
        "answer": real_answer
    }

datasets = {
    "length_4_digit": [],
    "length_5_digit": [],
    "negatives": [],
    "decimals": [],
    "unseen_operator_division": []
}

# 1. Generate 4-Digit (Length Scaling)
for _ in range(100):
    a, b = random.randint(1000, 9999), random.randint(1000, 9999)
    op = random.choice(['+', '-'])
    expr = f"{a} {op} {b}"
    datasets["length_4_digit"].append(format_problem(expr, eval(expr)))

# 2. Generate 5-Digit (Extreme Length Scaling)
for _ in range(100):
    a, b = random.randint(10000, 99999), random.randint(10000, 99999)
    op = random.choice(['+', '-'])
    expr = f"{a} {op} {b}"
    datasets["length_5_digit"].append(format_problem(expr, eval(expr)))

# 3. Generate Negatives (Format Shifting)
for _ in range(100):
    a, b = random.randint(-999, -1), random.randint(1, 999)
    op = random.choice(['+', '-'])
    expr = f"{a} {op} {b}"
    datasets["negatives"].append(format_problem(expr, eval(expr)))

# 4. Generate Decimals (Format Shifting)
for _ in range(100):
    a = round(random.uniform(1.0, 100.0), 2)
    b = round(random.uniform(1.0, 100.0), 2)
    op = random.choice(['+', '-'])
    expr = f"{a} {op} {b}"
    datasets["decimals"].append(format_problem(expr, round(eval(expr), 2)))

# 5. Unseen Operator (Division)
for _ in range(100):
    a = random.randint(10, 100)
    b = random.randint(1, 10) # Avoid division by zero
    expr = f"{a} / {b}"
    datasets["unseen_operator_division"].append(format_problem(expr, round(eval(expr), 2)))

# Save all datasets locally
for name, data in datasets.items():
    file_path = os.path.join(ood_dir, f"{name}.jsonl")
    with open(file_path, 'w') as f:
        for item in data:
            f.write(json.dumps(item) + '\n')
    print(f"✅ Saved {len(data)} problems to {file_path}")

print("\n🎉 All OOD Stress Test datasets generated successfully!")

✅ Saved 100 problems to data/ood_tests/length_4_digit.jsonl
✅ Saved 100 problems to data/ood_tests/length_5_digit.jsonl
✅ Saved 100 problems to data/ood_tests/negatives.jsonl
✅ Saved 100 problems to data/ood_tests/decimals.jsonl
✅ Saved 100 problems to data/ood_tests/unseen_operator_division.jsonl

🎉 All OOD Stress Test datasets generated successfully!


In [10]:
!cat instructor/final_project/arithmetic_llm/run_interactive.py | head -n 35

#!/usr/bin/env python3
"""Command-line interface for interactive arithmetic solving."""

import argparse
from .interactive_solver import InteractiveArithmeticSolver


def main():
    """Run interactive arithmetic solver from command line."""
    parser = argparse.ArgumentParser(
        description="Interactive arithmetic problem solver using trained LLM"
    )
    
    # Required arguments
    parser.add_argument(
        "--model-path",
        type=str,
        required=True,
        help="Path to instruction-tuned model checkpoint"
    )
    
    parser.add_argument(
        "--tokenizer-path",
        type=str,
        required=True,
        help="Path to tokenizer directory"
    )
    
    parser.add_argument(
        "--device",
        type=str,
        default="auto",
        help="Device for inference: 'cuda', 'mps', 'cpu', or 'auto' (default: auto)"
    )
    


In [14]:
!poetry env info --path

/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10


In [17]:
# 4-Digit Test
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260303_013325_759230/best_model.pt \
  --tokenizer-path data/tokenizer \
  --num-range 1000 9999 \
  --num-samples 100


MODEL EVALUATION

Model: models/instruction_20260303_013325_759230/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 100
  Max depth: 5
  Number range: 1000 to 9999
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 100 test expressions...
Generated 100 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/100 samples
Evaluated 2/100 samples
Evaluated 3/100 samples
Evaluated 4/100 samples
Evaluated 5/100 samples
Evaluated 6/100 samples
Evaluated 7/100 samples
Evaluated 8/100 samples
Evaluated 9/100 samples
Evaluated 10/100 samples
Evaluated 11/100 samples
Evaluated 12/100 samples
Evaluated 13/100 samples
Evaluated 14/100 samples
Evaluated 15/100 samples
Evaluated 16/100 samples
Evaluated 17/100 samples
Evaluated 18/100 samples
Evaluated 19/100 samples
Evaluated 20/100 samples
Evaluated 21/100 sam

In [2]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260303_013325_759230/best_model.pt \
  --tokenizer-path data/tokenizer \
  --num-range 10000 99999 \
  --num-samples 100


MODEL EVALUATION

Model: models/instruction_20260303_013325_759230/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 100
  Max depth: 5
  Number range: 10000 to 99999
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 100 test expressions...
Generated 100 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/100 samples
Evaluated 2/100 samples
Evaluated 3/100 samples
Evaluated 4/100 samples
Evaluated 5/100 samples
Evaluated 6/100 samples
Evaluated 7/100 samples
Evaluated 8/100 samples
Evaluated 9/100 samples
Evaluated 10/100 samples
Evaluated 11/100 samples
Evaluated 12/100 samples
Evaluated 13/100 samples
Evaluated 14/100 samples
Evaluated 15/100 samples
Evaluated 16/100 samples
Evaluated 17/100 samples
Evaluated 18/100 samples
Evaluated 19/100 samples
Evaluated 20/100 samples
Evaluated 21/100 s

In [3]:
!poetry run python -m instructor.final_project.arithmetic_llm.run_evaluation \
  --model-path models/instruction_20260303_013325_759230/best_model.pt \
  --tokenizer-path data/tokenizer \
  --num-range -999 999 \
  --num-samples 100


MODEL EVALUATION

Model: models/instruction_20260303_013325_759230/best_model.pt
Tokenizer: data/tokenizer
Device: cuda

Evaluation Configuration:
  Test samples: 100
  Max depth: 5
  Number range: -999 to 999
  Batch size: 1
  Max generation length: 512
  Output directory: evaluation_results

Loading model and tokenizer...
Model loaded successfully!

Starting evaluation...
Generating 100 test expressions...
Generated 21 valid test expressions
Evaluating model with batch size 1...
Evaluated 1/21 samples
Evaluated 2/21 samples
Evaluated 3/21 samples
Evaluated 4/21 samples
Evaluated 5/21 samples
Evaluated 6/21 samples
Evaluated 7/21 samples
Evaluated 8/21 samples
Evaluated 9/21 samples
Evaluated 10/21 samples
Evaluated 11/21 samples
Evaluated 12/21 samples
Evaluated 13/21 samples
Evaluated 14/21 samples
Evaluated 15/21 samples
Evaluated 16/21 samples
Evaluated 17/21 samples
Evaluated 18/21 samples
Evaluated 19/21 samples
Evaluated 20/21 samples
Evaluated 21/21 samples

Evaluation result

In [14]:
import sys
import os
import json
import re
import io
from contextlib import redirect_stdout
import torch

# 1. Environment Injection
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the stable baseline model
model_path = "models/instruction_20260302_162638_395697/best_model.pt"
tokenizer_path = "data/tokenizer"

print("Loading Model for Prompt Optimization Test...")
solver = InteractiveArithmeticSolver(
    model_path=model_path, 
    tokenizer_path=tokenizer_path, 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The Few-Shot Prompt Strategy
# We use ONLY known tokens to prime the model's attention mechanism
few_shot_primer = "1000 + 1000 = 2000 <think> Step 1 : 1000 + 1000 = 2000 Expression now : 2000 </think> Final Result : 2000 "

file_path = "data/ood_tests/length_4_digit.jsonl"
correct_count = 0
total_count = 0

print("\nRunning Few-Shot Prompting on 4-Digit OOD Data...")

with open(file_path, 'r') as f:
    for line in f:
        data = json.loads(line)
        original_expr = data['expression']
        true_ans = str(data['answer'])
        
        # Injecting the primer right before the expression
        primed_expr = few_shot_primer + original_expr
        
        trap = io.StringIO()
        with redirect_stdout(trap):
            try:
                solver.solve(primed_expr)
            except Exception:
                pass # Skip if the primer completely crashes the tokenizer
        
        model_output = trap.getvalue()
        
        # Extract the final result
        match = re.search(r"Final Result:\s*([^\s]+)", model_output)
        model_ans = match.group(1) if match else "PARSE_ERROR"
        
        if model_ans == true_ans:
            correct_count += 1
        total_count += 1

# 4. Results
if total_count > 0:
    accuracy = (correct_count / total_count) * 100
    print(f"\n========================================")
    print(f"ZERO-SHOT 4-DIGIT ACCURACY: 0.00%")
    print(f"FEW-SHOT 4-DIGIT ACCURACY:  {accuracy:.2f}%")
    print(f"========================================")
else:
    print("Could not process any problems.")

Loading Model for Prompt Optimization Test...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Running Few-Shot Prompting on 4-Digit OOD Data...

ZERO-SHOT 4-DIGIT ACCURACY: 0.00%
FEW-SHOT 4-DIGIT ACCURACY:  0.00%


In [15]:
import sys
import os
import io
import re
from contextlib import redirect_stdout
import torch

# 1. Environment Injection
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the baseline model
model_path = "models/instruction_20260302_162638_395697/best_model.pt"
tokenizer_path = "data/tokenizer"

print("Loading Model for A/B Prompt Optimization Test...")
solver = InteractiveArithmeticSolver(
    model_path=model_path, 
    tokenizer_path=tokenizer_path, 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. Our 3-Shot Primer (The "Cheat Sheet")
primer = (
    "Evaluate : 5 - 10 <think> Step 1 : 5 - 10 = - 5 Expression now : - 5 </think> Final Result : - 5 "
    "Evaluate : - 2 + 8 <think> Step 1 : - 2 + 8 = 6 Expression now : 6 </think> Final Result : 6 "
    "Evaluate : - 4 - 3 <think> Step 1 : - 4 - 3 = - 7 Expression now : - 7 </think> Final Result : - 7 "
)

# 4. The Test Dataset (20 OOD Negative Problems)
test_problems = [
    ("3 - 8", "-5"), ("-4 + 10", "6"), ("-2 - 5", "-7"), ("10 - 15", "-5"), 
    ("-6 + 2", "-4"), ("1 - 9", "-8"), ("-5 - 5", "-10"), ("7 - 12", "-5"),
    ("-8 + 14", "6"), ("-1 - 2", "-3"), ("4 - 11", "-7"), ("-9 + 3", "-6"),
    ("-3 - 6", "-9"), ("8 - 20", "-12"), ("-7 + 15", "8"), ("2 - 6", "-4"),
    ("-10 + 5", "-5"), ("-4 - 8", "-12"), ("5 - 18", "-13"), ("-6 + 12", "6")
]

def extract_answer(output_text):
    match = re.search(r"Final Result\s*:\s*([^\s]+)", output_text)
    return match.group(1) if match else "ERROR"

zero_shot_correct = 0
multi_shot_correct = 0

print("\nRunning Direct Comparison...")

for expr, true_ans in test_problems:
    # A. Zero-Shot Test (Just the expression)
    trap_zero = io.StringIO()
    with redirect_stdout(trap_zero):
        solver.solve(expr)
    ans_zero = extract_answer(trap_zero.getvalue())
    if ans_zero == true_ans: zero_shot_correct += 1

    # B. Multi-Shot Test (Primer + the expression)
    primed_expr = primer + f"Evaluate : {expr}"
    trap_multi = io.StringIO()
    with redirect_stdout(trap_multi):
        solver.solve(primed_expr)
    ans_multi = extract_answer(trap_multi.getvalue())
    if ans_multi == true_ans: multi_shot_correct += 1

# 5. The Final Reveal
zero_acc = (zero_shot_correct / len(test_problems)) * 100
multi_acc = (multi_shot_correct / len(test_problems)) * 100

print(f"\n{'='*40}")
print(f"ZERO-SHOT ACCURACY:  {zero_acc:.2f}%")
print(f"MULTI-SHOT ACCURACY: {multi_acc:.2f}%")
print(f"{'='*40}")

if multi_acc > zero_acc:
    print(f"SUCCESS! You rescued the model's reasoning by +{multi_acc - zero_acc:.2f}%!")
else:
    print("The model's architectural limits could not be rescued by prompting.")

Loading Model for A/B Prompt Optimization Test...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Running Direct Comparison...

ZERO-SHOT ACCURACY:  0.00%
MULTI-SHOT ACCURACY: 0.00%
The model's architectural limits could not be rescued by prompting.


In [16]:
!poetry run python -m instructor.final_project.arithmetic_llm.print_token_table --tokenizer_path data/tokenizer/tokenizer.pkl > tokens.csv

In [17]:
!poetry run python -m instructor.final_project.arithmetic_llm.check_sequence_lengths \
  --corpus-path data/instruction_corpus.txt \
  --tokenizer-path data/tokenizer

Auto-detected corpus type: instruction
Loading tokenizer from: data/tokenizer

Analyzing corpus: data/instruction_corpus.txt
Corpus type: instruction
Analyzing: all lines

SEQUENCE LENGTH ANALYSIS

Total sequences analyzed: 40000

Basic Statistics:
  Min length:         12 tokens
  Max length:        687 tokens
  Mean length:     165.4 tokens
  Median length:   125.0 tokens
  Std deviation:   160.0 tokens

Percentiles:
   50.0th percentile:    125 tokens
   75.0th percentile:    251 tokens
   90.0th percentile:    403 tokens
   95.0th percentile:    471 tokens
   99.0th percentile:    592 tokens
   99.5th percentile:    606 tokens
  100.0th percentile:    687 tokens

Coverage by max_seq_length:
  max_seq_length=  64:  15481/40000 ( 38.7%) | 24519 truncated
  max_seq_length= 128:  20747/40000 ( 51.9%) | 19253 truncated
  max_seq_length= 192:  23906/40000 ( 59.8%) | 16094 truncated
  max_seq_length= 256:  30421/40000 ( 76.1%) |  9579 truncated
  max_seq_length= 384:  34490/40000 ( 86.2%)

In [6]:
%%writefile generate_demo.py
import torch
from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# Load the Baseline Model
model_path = "models/instruction_20260302_162638_395697/best_model.pt"
tokenizer_path = "data/tokenizer"

solver = InteractiveArithmeticSolver(
    model_path=model_path, 
    tokenizer_path=tokenizer_path, 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

demo_prompts = [
    ("SCENARIO 1: IN-DISTRIBUTION SUCCESS", "15 + 4"),
    ("SCENARIO 2: ATTENTION OVERLOAD (OOD)", "4035 + 8268"),
    ("SCENARIO 3: TOKENIZER WALL (OOD)", "10 / 2")
]

print("\n" + "="*60)
print("ARITHMETIC LLM: REASONING DEMO")
print("="*60)

for scenario_name, expression in demo_prompts:
    print(f"\n{scenario_name}")
    print(f"User Input: Evaluate: {expression}")
    print("-" * 40)
    print("Model Output:\n")
    
    # THE FIX: We actually capture the returned text and print it!
    try:
        result = solver.solve(expression)
        print(result)
    except Exception as e:
        print(f"[Model Crashed or Returned Error]: {e}")
    
    print("="*60)

Overwriting generate_demo.py


In [7]:
!poetry run python generate_demo.py

Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

ARITHMETIC LLM: REASONING DEMO

SCENARIO 1: IN-DISTRIBUTION SUCCESS
User Input: Evaluate: 15 + 4
----------------------------------------
Model Output:

Evaluate : 15 + 4 <think> <think> Step 1 : 15 + 4 = 19 Expression now : 19 </think> Final Result : 19

SCENARIO 2: ATTENTION OVERLOAD (OOD)
User Input: Evaluate: 4035 + 8268
----------------------------------------
Model Output:

Evaluate : 4035 + 8268 <think> <think> Step 1 : 35 + 116 = 78 Expression now : 78 </think> Final Result : 93

SCENARIO 3: TOKENIZER WALL (OOD)
User Input: Evaluate: 10 / 2
----------------------------------------
Model Output:

Evaluate : 10  2 <think> Step 1 : 10 + 2 = 12 Expression now : 12 </think> Final Result : 12


In [14]:
import json
import re
import sys
import os
import torch

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The File We Want to Grade (Change this name to test the other files!)
test_file = "data/ood_tests/length_4_digit.jsonl"

total = 0
parse_success = 0
exact_match = 0

print(f"\nEvaluating OOD File: {test_file}")
print("-" * 50)

# 4. Grade the Model
with open(test_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        expr = data['expression']
        true_ans = str(data['answer'])
        
        # Get model output
        try:
            output = solver.solve(expr)
        except Exception:
            output = "" # If it crashes completely
            
        total += 1
        
        # Check Parse Rate (Did it format the logic properly?)
        parsed_ans = None
        if "<think>" in output and "Final Result :" in output:
            parse_success += 1
            
            # Extract the final answer it gave
            match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
            if match:
                parsed_ans = match.group(1)
        
        # Check Accuracy (Did the math match?)
        if parsed_ans == true_ans:
            exact_match += 1

# 5. The Final Results
print(f"Total Samples Evaluated: {total}")
print(f"Parse Success Rate:      {(parse_success/total)*100:.2f}%")
print(f"Exact Match Accuracy:    {(exact_match/total)*100:.2f}%")
print("-" * 50)

Loading Model...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Evaluating OOD File: data/ood_tests/length_4_digit.jsonl
--------------------------------------------------
Total Samples Evaluated: 100
Parse Success Rate:      71.00%
Exact Match Accuracy:    0.00%
--------------------------------------------------


In [15]:
import json
import re
import sys
import os
import torch

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The File We Want to Grade (Change this name to test the other files!)
test_file = "data/ood_tests/decimals.jsonl"

total = 0
parse_success = 0
exact_match = 0

print(f"\nEvaluating OOD File: {test_file}")
print("-" * 50)

# 4. Grade the Model
with open(test_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        expr = data['expression']
        true_ans = str(data['answer'])
        
        # Get model output
        try:
            output = solver.solve(expr)
        except Exception:
            output = "" # If it crashes completely
            
        total += 1
        
        # Check Parse Rate (Did it format the logic properly?)
        parsed_ans = None
        if "<think>" in output and "Final Result :" in output:
            parse_success += 1
            
            # Extract the final answer it gave
            match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
            if match:
                parsed_ans = match.group(1)
        
        # Check Accuracy (Did the math match?)
        if parsed_ans == true_ans:
            exact_match += 1

# 5. The Final Results
print(f"Total Samples Evaluated: {total}")
print(f"Parse Success Rate:      {(parse_success/total)*100:.2f}%")
print(f"Exact Match Accuracy:    {(exact_match/total)*100:.2f}%")
print("-" * 50)

Loading Model...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Evaluating OOD File: data/ood_tests/decimals.jsonl
--------------------------------------------------
Total Samples Evaluated: 100
Parse Success Rate:      26.00%
Exact Match Accuracy:    0.00%
--------------------------------------------------


In [16]:
import json
import re
import sys
import os
import torch

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The File We Want to Grade (Change this name to test the other files!)
test_file = "data/ood_tests/negatives.jsonl"

total = 0
parse_success = 0
exact_match = 0

print(f"\nEvaluating OOD File: {test_file}")
print("-" * 50)

# 4. Grade the Model
with open(test_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        expr = data['expression']
        true_ans = str(data['answer'])
        
        # Get model output
        try:
            output = solver.solve(expr)
        except Exception:
            output = "" # If it crashes completely
            
        total += 1
        
        # Check Parse Rate (Did it format the logic properly?)
        parsed_ans = None
        if "<think>" in output and "Final Result :" in output:
            parse_success += 1
            
            # Extract the final answer it gave
            match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
            if match:
                parsed_ans = match.group(1)
        
        # Check Accuracy (Did the math match?)
        if parsed_ans == true_ans:
            exact_match += 1

# 5. The Final Results
print(f"Total Samples Evaluated: {total}")
print(f"Parse Success Rate:      {(parse_success/total)*100:.2f}%")
print(f"Exact Match Accuracy:    {(exact_match/total)*100:.2f}%")
print("-" * 50)

Loading Model...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Evaluating OOD File: data/ood_tests/negatives.jsonl
--------------------------------------------------
Total Samples Evaluated: 100
Parse Success Rate:      100.00%
Exact Match Accuracy:    0.00%
--------------------------------------------------


In [17]:
import json
import re
import sys
import os
import torch

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The File We Want to Grade (Change this name to test the other files!)
test_file = "data/ood_tests/unseen_operator_division.jsonl"

total = 0
parse_success = 0
exact_match = 0

print(f"\nEvaluating OOD File: {test_file}")
print("-" * 50)

# 4. Grade the Model
with open(test_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        expr = data['expression']
        true_ans = str(data['answer'])
        
        # Get model output
        try:
            output = solver.solve(expr)
        except Exception:
            output = "" # If it crashes completely
            
        total += 1
        
        # Check Parse Rate (Did it format the logic properly?)
        parsed_ans = None
        if "<think>" in output and "Final Result :" in output:
            parse_success += 1
            
            # Extract the final answer it gave
            match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
            if match:
                parsed_ans = match.group(1)
        
        # Check Accuracy (Did the math match?)
        if parsed_ans == true_ans:
            exact_match += 1

# 5. The Final Results
print(f"Total Samples Evaluated: {total}")
print(f"Parse Success Rate:      {(parse_success/total)*100:.2f}%")
print(f"Exact Match Accuracy:    {(exact_match/total)*100:.2f}%")
print("-" * 50)

Loading Model...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Evaluating OOD File: data/ood_tests/unseen_operator_division.jsonl
--------------------------------------------------
Total Samples Evaluated: 100
Parse Success Rate:      77.00%
Exact Match Accuracy:    0.00%
--------------------------------------------------


In [18]:
import json
import re
import sys
import os
import torch

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The File We Want to Grade (Change this name to test the other files!)
test_file = "data/ood_tests/length_5_digit.jsonl"

total = 0
parse_success = 0
exact_match = 0

print(f"\nEvaluating OOD File: {test_file}")
print("-" * 50)

# 4. Grade the Model
with open(test_file, 'r') as f:
    for line in f:
        data = json.loads(line)
        expr = data['expression']
        true_ans = str(data['answer'])
        
        # Get model output
        try:
            output = solver.solve(expr)
        except Exception:
            output = "" # If it crashes completely
            
        total += 1
        
        # Check Parse Rate (Did it format the logic properly?)
        parsed_ans = None
        if "<think>" in output and "Final Result :" in output:
            parse_success += 1
            
            # Extract the final answer it gave
            match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
            if match:
                parsed_ans = match.group(1)
        
        # Check Accuracy (Did the math match?)
        if parsed_ans == true_ans:
            exact_match += 1

# 5. The Final Results
print(f"Total Samples Evaluated: {total}")
print(f"Parse Success Rate:      {(parse_success/total)*100:.2f}%")
print(f"Exact Match Accuracy:    {(exact_match/total)*100:.2f}%")
print("-" * 50)

Loading Model...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

Evaluating OOD File: data/ood_tests/length_5_digit.jsonl
--------------------------------------------------
Total Samples Evaluated: 100
Parse Success Rate:      18.00%
Exact Match Accuracy:    0.00%
--------------------------------------------------


In [21]:
import json
import sys
import os
import torch
import numpy as np

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Baseline Model
print("Loading Model and Tokenizer...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)
tokenizer = solver.tokenizer

# 3. The Files to Evaluate
ood_files = {
    "4-Digit": "data/ood_tests/length_4_digit.jsonl",
    "5-Digit": "data/ood_tests/length_5_digit.jsonl",
    "Decimals": "data/ood_tests/decimals.jsonl",
    "Negatives": "data/ood_tests/negatives.jsonl",
    "Division": "data/ood_tests/unseen_operator_division.jsonl"
}

print("\n" + "="*50)
print("EXTRACTING TRUE GENERATION LENGTHS")
print("="*50)

for category, filepath in ood_files.items():
    lengths = []
    
    try:
        with open(filepath, 'r') as f:
            for line in f:
                data = json.loads(line)
                expr = data['expression']
                
                # Get model output
                try:
                    output = solver.solve(expr)
                except Exception:
                    output = ""
                
                # Count the exact tokens generated
                if hasattr(tokenizer, 'encode'):
                    encoded = tokenizer.encode(output)
                    token_count = len(encoded.ids if hasattr(encoded, 'ids') else encoded)
                else:
                    token_count = len(tokenizer(output)['input_ids'])
                    
                lengths.append(token_count)
                
        avg_length = np.mean(lengths)
        print(f"{category:<15} | True Avg Tokens: {avg_length:.2f}")
        
    except FileNotFoundError:
        print(f"{category:<15} | ERROR: File not found.")

print("="*50)

Loading Model and Tokenizer...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

EXTRACTING TRUE GENERATION LENGTHS
4-Digit         | True Avg Tokens: 27.52
5-Digit         | True Avg Tokens: 23.26
Decimals        | True Avg Tokens: 23.02
Negatives       | True Avg Tokens: 31.70
Division        | True Avg Tokens: 13.83


In [25]:
import json
import sys
import os
import torch
import re

# 1. Environment Setup
VENV_PATH = "/home/jupyter/.cache/pypoetry/virtualenvs/stat359-su25-t0ZXeGPH-py3.10" 
LIB_PATH = os.path.join(VENV_PATH, "lib/python3.10/site-packages")

if LIB_PATH not in sys.path:
    sys.path.append(LIB_PATH)
    sys.path.append(os.getcwd())

from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# 2. Load the Model
print("Loading Model for Prompt Optimization Test...")
solver = InteractiveArithmeticSolver(
    model_path="models/instruction_20260302_162638_395697/best_model.pt", 
    tokenizer_path="data/tokenizer", 
    device="cuda" if torch.cuda.is_available() else "cpu"
)

# 3. The Extreme Few-Shot Prompt
# We are explicitly teaching it the rules of negative numbers
few_shot_context = """Here are examples of how to solve math with negative numbers:

Evaluate: -5 + 3
<think>
Step 1: -5 + 3 = -2
Expression now: -2
</think>
Final Result: -2

Evaluate: 4 - 9
<think>
Step 1: 4 - 9 = -5
Expression now: -5
</think>
Final Result: -5

Evaluate: -2 - 4
<think>
Step 1: -2 - 4 = -6
Expression now: -6
</think>
Final Result: -6

Now solve this:
Evaluate: """

print("\n" + "="*50)
print("RUNNING EXTREME FEW-SHOT OPTIMIZATION")
print("="*50)

test_file = "data/ood_tests/negatives.jsonl"
total = 0
exact_match = 0

with open(test_file, 'r') as f:
    # Testing the first 50 samples to save time
    for i, line in enumerate(f):
        if i >= 50: break 
        
        data = json.loads(line)
        # We strip the "Evaluate: " from the original expression so we can append it cleanly to our prompt
        raw_expr = data['expression'].replace("Evaluate: ", "").strip()
        true_ans = str(data['answer'])
        
        # Combine the context with the new question
        full_prompt = few_shot_context + raw_expr
        
        try:
            output = solver.solve(full_prompt)
        except Exception:
            output = ""
            
        total += 1
        
        # Check Accuracy
        parsed_ans = None
        match = re.search(r"Final Result\s*:\s*([^\s]+)", output)
        if match:
            parsed_ans = match.group(1)
            
        if parsed_ans == true_ans:
            exact_match += 1

accuracy = (exact_match/total)*100
print(f"Few-Shot Prompted Accuracy on Negatives: {accuracy:.2f}%")
print("="*50)

Loading Model for Prompt Optimization Test...
Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

RUNNING EXTREME FEW-SHOT OPTIMIZATION
Few-Shot Prompted Accuracy on Negatives: 0.00%


In [26]:
import json, re, sys, os
from instructor.final_project.arithmetic_llm.interactive_solver import InteractiveArithmeticSolver

# Load Model
solver = InteractiveArithmeticSolver(model_path="models/instruction_20260302_162638_395697/best_model.pt", tokenizer_path="data/tokenizer")

few_shot_context = "Evaluate: -5 + 3\n<think>\nStep 1: -5 + 3 = -2\nExpression now: -2\n</think>\nFinal Result: -2\n\nNow solve: Evaluate: "

results = {"Zero-Shot": {"acc": 0, "parse": 0}, "Few-Shot": {"acc": 0, "parse": 0}}
test_file = "data/ood_tests/negatives.jsonl"

with open(test_file, 'r') as f:
    samples = [json.loads(line) for line in f][:20]

for name in ["Zero-Shot", "Few-Shot"]:
    correct, parsed = 0, 0
    for s in samples:
        prompt = s['expression'] if name == "Zero-Shot" else few_shot_context + s['expression'].replace("Evaluate: ", "")
        out = solver.solve(prompt)
        if "<think>" in out and "Final Result :" in out:
            parsed += 1
            ans = re.search(r"Final Result\s*:\s*([^\s]+)", out)
            if ans and ans.group(1) == str(s['answer']):
                correct += 1
    results[name]["acc"], results[name]["parse"] = (correct/20)*100, (parsed/20)*100

print(f"\nFINAL VERIFIED DATA:\nZero-Shot: {results['Zero-Shot']}\nFew-Shot: {results['Few-Shot']}")

Loading model and tokenizer...
Model loaded successfully on cuda
Vocabulary size: 278

FINAL VERIFIED DATA:
Zero-Shot: {'acc': 0.0, 'parse': 100.0}
Few-Shot: {'acc': 0.0, 'parse': 100.0}
